# SETUP

In [ ]:
# Read libraries
import pandas as pd
import numpy as np
import os
import re
import networkx as nx
from tqdm import tqdm



# ML libraries
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from langdetect import detect, DetectorFactory
import torch

from ortools.graph.python import min_cost_flow

# Open file in read mode

taxonomy_path = os.path.join("data", "taxonomy.txt")
count_of_products_per_level1_path = os.path.join("data", "count_of_products_per_level1.csv")
data_path = os.path.join("data", "ensae_export_without_l1.parquet")

## Read categories files

In [4]:
# --------- Lire la taxonomy depuis un fichier txt ----------
# Assumons que le fichier s'appelle "taxonomy.txt"
# Format attendu : id_path <tab> category_path
df_taxonomy = pd.read_csv(taxonomy_path, sep='\t', header=None, names=['id_path', 'category_path'])

# Nettoyage
df_taxonomy['category_path'] = df_taxonomy['category_path'].str.strip()
df_taxonomy['id_path'] = df_taxonomy['id_path'].str.strip()

# Construire le graphe dirigé de la taxonomie
G = nx.DiGraph()
root = "ROOT"  # racine commune
G.add_node(root)

for path in df_taxonomy['category_path']:
    parts = [p.strip() for p in path.split(">")]
    if parts:  # relier le level_1 à la racine
        G.add_edge(root, parts[0])
    for i in range(len(parts)-1):
        parent = parts[i]
        child = parts[i+1]
        G.add_edge(parent, child)

# Identifier les level_1
level_1_nodes = [p.split(">")[0].strip() for p in df_taxonomy['category_path']]
level_1_nodes = list(set(level_1_nodes))

print("Level 1 categories:", level_1_nodes)

Level 1 categories: ['Airlines', 'cameras & optics', 'toys & games', 'food, beverages & tobacco', 'luggage & bags', 'animals & pet supplies', 'Communication', 'baby & toddler', 'media', 'Employment', 'home & garden', 'furniture', 'health & beauty', 'office supplies', 'apparel & accessories', 'Gaming/Gambling', 'sporting goods', 'software', 'Goods', 'Ground/Cruises/Packages', 'religious & ceremonial', 'Real Estate', 'Travel', 'Car Rental', 'Services', 'Finance Services', 'Hotels/Resorts', 'business & industrial', 'vehicles & parts', 'electronics', 'hardware', 'Dating', 'arts & entertainment', 'mature']


In [8]:
# Extract level 1 category counts

df_categories_count = pd.read_csv(count_of_products_per_level1_path)
df_categories_count

# Keep only level 1 categories present in both dataframes
common_level1 = set(level_1_nodes).intersection(set(df_categories_count['level_1_name']))
df_categories_count = df_categories_count[df_categories_count['level_1_name'].isin(common_level1)]

print(f" Number of level 1 categories in taxonomy: {len(level_1_nodes)}")
print(f" Number of level 1 categories in count_of_products_per_level1.csv: {len(df_categories_count)}")
print(f"Dropping {len(level_1_nodes) - len(common_level1)} level 1 categories from taxonomy that are not in count_of_products_per_level1.csv")

 Number of level 1 categories in taxonomy: 34
 Number of level 1 categories in count_of_products_per_level1.csv: 21
Dropping 13 level 1 categories from taxonomy that are not in count_of_products_per_level1.csv


In [9]:
print("Number of products : ", df_categories_count["count"].sum())

Number of products :  128253


## Read catalog files

In [10]:
df_catalog = pd.read_parquet(data_path, engine='pyarrow')

## Preprocessing df_catalog

In [11]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder

def preprocess_for_nlp(df, text_cols=['title', 'description'], brand_col='brand', id_col='hashed_external_id' , price_col = 'sale_price'):
    """
    Preprocess products DataFrame for NLP tasks:
    - Concatenate text columns into a single 'text' column
    - Encode brand as integer
    - Clean text: lowercasing, remove punctuation, multiple spaces
    - Keep hashed_external_id for final output

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    text_cols : list of str
        Columns to concatenate for text
    brand_col : str
        Column to encode as integer
    id_col : str
        Column to keep for external mapping
    price_col : str
        Column for price information

    Returns
    -------
    df_preprocessed : pd.DataFrame
        DataFrame with columns: 'hashed_external_id', 'text', brand_col (encoded)
    label_enc : LabelEncoder
        Fitted LabelEncoder for the brand column
    """
    df = df.copy()

    # Fill missing values for text columns
    for col in text_cols:
        df[col] = df[col].fillna('')

    # Concatenate text columns
    df['text'] = df[text_cols].agg(' '.join, axis=1)

    # Clean text
    def clean_text(s):
        s = s.lower()
        s = re.sub(r'\s+', ' ', s)      # multiple spaces -> single space
        s = re.sub(r'[^\w\s]', '', s)   # remove punctuation
        return s.strip()

    df['text'] = df['text'].apply(clean_text)

    # Encode brand as integer
    if brand_col in df.columns:
        df[brand_col] = df[brand_col].fillna('Unknown')
        label_enc = LabelEncoder()
        df[brand_col + '_encoded'] = label_enc.fit_transform(df[brand_col])
    else:
        label_enc = None

    # Fill na for price column
    if price_col in df.columns:
        df[price_col] = df[price_col].astype(float)
        df[price_col] = df[price_col].fillna(df[price_col].median())

    # Keep hashed_external_id
    columns_to_keep = [id_col, 'text' , price_col]
    if label_enc:
        columns_to_keep.append(brand_col + '_encoded')

    return df[columns_to_keep], label_enc

# --------- Usage ---------
df_nlp, brand_encoder = preprocess_for_nlp(
    df_catalog,
    text_cols=['title','description','brand'],
    brand_col='brand',
    id_col='hashed_external_id',
    price_col = 'sale_price'
)

df_nlp.head()

,hashed_external_id,text,sale_price,brand_encoded
0,-2772291400701920348,the hoodoo tarot a divination deck and guidebo...,35.00,0
1,-4184851053829790189,disney villains tarot deck and guidebook movie...,24.99,0
2,-8778697834751578524,easy tarot created especially for beginners th...,19.95,0
3,-3541475158234224984,the proudest blue a story of hijab and family ...,17.99,0
4,-2529310467283008815,the crystal magic tarot understand and control...,24.95,0


### check language distribution

In [16]:
# Pour rendre les résultats reproductibles
DetectorFactory.seed = 0

def detect_language(text):
    """
    Detect the language of a given text.
    
    Parameters
    ----------
    text : str
        Input text
    
    Returns
    -------
    lang_code : str
        ISO 639-1 language code (e.g., 'en' for English)
    """
    try:
        return detect(text)
    except:
        return "unknown"

language_df_level1_clean = df_level1_clean["concatenated_text"].apply(detect_language)
language_df_nlp = df_nlp["text"].apply(detect_language)

In [17]:
language_df_level1_clean.value_counts()
language_df_nlp.value_counts()

text
en    128107
es        37
fr        30
it        20
ca        16
no        10
af         8
nl         7
da         5
ro         4
tl         2
sv         2
lt         2
cy         2
et         1
Name: count, dtype: int64

the vast majority of the text seems to be in English, with some other languages mixed in.

Decide to drop non-English entries for simplicity.


In [ ]:
language_df_level1_clean.value_counts().to_csv("data/language_distribution_level1_clean_count.csv", index=True)
language_df_nlp.value_counts().to_csv("data/language_distribution_nlp_count.csv", index=True)

language_df_level1_clean.to_csv("data/language_distribution_level1_clean.csv",index=True,columns=["idx","language"])
language_df_nlp.to_csv("data/language_df_nlp.csv" , index=True , columns=["idx","language"])

In [7]:
## drop non-english entries

language_df_nlp = pd.read_csv("data/language_df_nlp.csv")
df_nlp = df_nlp[language_df_nlp["language"] == 'en']

# Generate embeddings with sentence transformers

In [ ]:
# ---------- 1. Choisir le device ----------
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

# ---------- 2. Charger le modèle ----------
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ---------- 3. Préparer les textes ----------
texts_catalog = df_nlp['text'].tolist()

# ---------- 4. Encode avec batch et tqdm ----------
def encode_texts(texts, batch_size=64):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        batch_emb = model.encode(
            batch_texts,
            convert_to_numpy=True,
            device=device,
            show_progress_bar=False
        )
        embeddings.append(batch_emb)
    return np.vstack(embeddings)

emb_catalog = encode_texts(texts_catalog, batch_size=64)


all_nodes = list(G.nodes())
all_nodes.remove("ROOT")

node_embeddings_raw = encode_texts(all_nodes, batch_size=64)
node_to_emb = dict(zip(all_nodes, node_embeddings_raw))

# 2. Calculer la moyenne des embeddings pour chaque branche de level_1
level_1_embeddings = []
valid_level_1_names = []

for lvl1 in df_categories_count["level_1_name"]:
    sub_hierarchy = list(nx.descendants(G, lvl1)) + [lvl1]
    
    # Récupérer les vecteurs de ces catégories
    vectors = [node_to_emb[node] for node in sub_hierarchy if node in node_to_emb]
    
    if vectors:
        branch_mean = np.mean(vectors, axis=0)
        level_1_embeddings.append(branch_mean)
        valid_level_1_names.append(lvl1)

emb_taxonomy = np.vstack(level_1_embeddings)

print("Forme finale de emb_taxonomy :", emb_taxonomy.shape)
print("Catalogue embeddings shape:", emb_catalog.shape)
print("Taxonomy embeddings shape:", emb_taxonomy.shape)

# ---------- 5. Sauvegarde compressée ----------
np.savez_compressed("data/emb_catalog.npz", emb_catalog)
np.savez_compressed("data/emb_taxonomy.npz", emb_taxonomy)

Using device: mps


100%|██████████| 85/85 [00:07<00:00, 10.82it/s]


Forme finale de emb_taxonomy : (21, 384)
Taxonomy embeddings shape: (21, 384)


## Save embeddings 

In [16]:
# Embedding without filtering the non-english language product 

emb_catalog = np.load("data/emb_catalog.npz")["arr_0"]
emb_taxonomy = np.load("data/emb_taxonomy.npz")["arr_0"]

there is brand that sell multiple products, so we will keep brand information for each product entry.

### KNN

In [17]:
# --------------------------
# 1. Compute similarity matrix between catalogue and taxonomy embedings
# --------------------------

similarity_matrix = cosine_similarity(emb_catalog, emb_taxonomy)


In [21]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# Noms des colonnes dans df_categories_count
COL_NAME_COUNT = 'count'           # Quota de chaque catégorie
COL_NAME_CATEGORY = 'level_1_name' # Nom de la catégorie

# Paramètres d'optimisation
TOP_K = 10              # Nombre de catégories candidates par produit
COST_MULTIPLIER = 100000  # Facteur de conversion float -> int

# ==============================================================================
# 2. VALIDATION DES DONNÉES
# ==============================================================================

def validate_inputs(similarity_matrix, df_categories_count):
    """Vérifie la cohérence des données d'entrée"""
    
    n_products, n_cols_sim = similarity_matrix.shape
    n_cats_df = len(df_categories_count)
    
    # Vérification des dimensions
    if n_cols_sim != n_cats_df:
        raise ValueError(
            f"❌ Incompatibilité : {n_cols_sim} colonnes dans la matrice "
            f"mais {n_cats_df} catégories dans le DataFrame"
        )
    
    # Extraction et vérification des quotas
    quotas = df_categories_count[COL_NAME_COUNT].values.astype(int)
    total_quotas = quotas.sum()
    
    print(f"📊 Données validées :")
    print(f"   • {n_products:,} produits à répartir")
    print(f"   • {n_cats_df} catégories disponibles")
    print(f"   • Somme des quotas : {total_quotas:,}")
    
    # Ajustement automatique si nécessaire
    diff = n_products - total_quotas
    if diff != 0:
        print(f"\n⚠️  Ajustement nécessaire : {abs(diff)} produit(s)")
        print(f"   → Modification du quota de '{df_categories_count.iloc[-1][COL_NAME_CATEGORY]}'")
        quotas[-1] += diff
        print(f"   → Nouveau total : {quotas.sum():,}")
    
    return n_products, n_cats_df, quotas


# ==============================================================================
# 3. CONSTRUCTION DU GRAPHE
# ==============================================================================

def build_flow_graph(similarity_matrix, quotas, n_products, n_cats, top_k=10):
    """
    Construit le graphe de flux min-cost max-flow
    
    Structure :
    SOURCE → [Produits] → [Catégories] → SINK
    """
    
    smcf = min_cost_flow.SimpleMinCostFlow()
    
    # Indices des nœuds
    SOURCE = 0
    SINK = n_products + n_cats + 1
    START_PROD = 1
    START_CAT = n_products + 1
    
    print(f"\n🔨 Construction du graphe...")
    
    # --- Étape 1 : SOURCE → Produits ---
    # Chaque produit reçoit exactement 1 unité de flux
    sources = [SOURCE] * n_products
    targets = list(range(START_PROD, START_PROD + n_products))
    capacities = [1] * n_products
    costs = [0] * n_products
    
    smcf.add_arcs_with_capacity_and_unit_cost(sources, targets, capacities, costs)
    print(f"   ✓ {n_products:,} arcs Source → Produits")
    
    # --- Étape 2 : Produits → Catégories (TOP-K) ---
    # Pour chaque produit, on ne garde que les K catégories les plus similaires
    print(f"   • Calcul des {top_k} meilleures catégories par produit...")
    
    # Optimisation : argpartition est plus rapide que argsort
    top_k_actual = min(top_k, n_cats)
    top_k_indices = np.argpartition(similarity_matrix, -top_k_actual, axis=1)[:, -top_k_actual:]
    
    u_list, v_list, cap_list, cost_list = [], [], [], []
    
    for i in range(n_products):
        candidates = top_k_indices[i]
        row_scores = similarity_matrix[i]
        
        for cat_idx in candidates:
            score = row_scores[cat_idx]
            
            # Coût = (1 - similarité) × multiplicateur
            # Plus la similarité est élevée, plus le coût est faible
            cost = int((1.0 - score) * COST_MULTIPLIER)
            
            u_list.append(START_PROD + i)
            v_list.append(START_CAT + int(cat_idx))
            cap_list.append(1)
            cost_list.append(cost)
    
    smcf.add_arcs_with_capacity_and_unit_cost(u_list, v_list, cap_list, cost_list)
    print(f"   ✓ {len(u_list):,} arcs Produits → Catégories (Top-{top_k_actual})")
    
    # --- Étape 3 : Catégories → SINK ---
    # Chaque catégorie peut recevoir au maximum son quota
    u_sink = [START_CAT + j for j in range(n_cats)]
    v_sink = [SINK] * n_cats
    cap_sink = [int(q) for q in quotas]
    cost_sink = [0] * n_cats
    
    smcf.add_arcs_with_capacity_and_unit_cost(u_sink, v_sink, cap_sink, cost_sink)
    print(f"   ✓ {n_cats} arcs Catégories → Sink")
    
    # --- Étape 4 : Définition du flux global ---
    smcf.set_node_supply(SOURCE, int(n_products))
    smcf.set_node_supply(SINK, -int(n_products))
    
    return smcf, START_PROD, START_CAT


# ==============================================================================
# 4. RÉSOLUTION ET EXTRACTION
# ==============================================================================

def solve_and_extract(smcf, n_products, start_prod, start_cat, cat_names):
    """Résout le problème et extrait les assignations"""
    
    print(f"\n🚀 Résolution du problème d'optimisation...")
    status = smcf.solve()
    
    if status != smcf.OPTIMAL:
        raise RuntimeError(
            "❌ Aucune solution optimale trouvée. "
            "Vérifiez que la somme des quotas égale le nombre de produits "
            "et que TOP_K est suffisamment grand."
        )
    
    print(f"✅ Solution optimale trouvée !")
    print(f"   • Coût total : {smcf.optimal_cost():,}")
    print(f"   • Similarité moyenne : {1 - (smcf.optimal_cost() / (n_products * COST_MULTIPLIER)):.4f}")
    
    # Extraction des assignations
    assignments = np.full(n_products, -1, dtype=int)
    
    for arc_id in range(smcf.num_arcs()):
        if smcf.flow(arc_id) > 0:
            u = smcf.tail(arc_id)
            v = smcf.head(arc_id)
            
            # On cherche les arcs Produit → Catégorie
            if start_prod <= u < start_cat:
                prod_idx = u - start_prod
                cat_idx = v - start_cat
                assignments[prod_idx] = cat_idx
    
    # Vérification
    if np.any(assignments == -1):
        n_unassigned = np.sum(assignments == -1)
        raise RuntimeError(f"❌ {n_unassigned} produits non assignés !")
    
    # Conversion en noms de catégories
    predicted_labels = [cat_names[i] for i in assignments]
    
    return assignments, predicted_labels


# ==============================================================================
# 5. FONCTION PRINCIPALE
# ==============================================================================

def assign_products_to_categories(similarity_matrix, df_categories_count, top_k=10):
    """
    Fonction principale pour assigner les produits aux catégories
    
    Args:
        similarity_matrix : np.array de shape (n_products, n_categories)
                           Matrice de similarité cosine
        df_categories_count : DataFrame avec colonnes 'level_1_name' et 'count'
        top_k : Nombre de catégories candidates par produit
    
    Returns:
        assignments : np.array des indices de catégories assignées
        predicted_labels : list des noms de catégories assignées
        stats : dict avec des statistiques sur la répartition
    """
    
    print("=" * 70)
    print("RÉPARTITION OPTIMALE - MIN-COST MAX-FLOW")
    print("=" * 70)
    
    # Validation
    n_products, n_cats, quotas = validate_inputs(similarity_matrix, df_categories_count)
    
    # Construction du graphe
    smcf, start_prod, start_cat = build_flow_graph(
        similarity_matrix, quotas, n_products, n_cats, top_k
    )
    
    # Résolution
    cat_names = df_categories_count[COL_NAME_CATEGORY].values
    assignments, predicted_labels = solve_and_extract(
        smcf, n_products, start_prod, start_cat, cat_names
    )
    
    # Statistiques
    unique, counts = np.unique(assignments, return_counts=True)
    stats = {
        'distribution': dict(zip(cat_names[unique], counts)),
        'quotas_respectes': np.allclose(counts, quotas[unique])
    }
    
    print(f"\n📈 Statistiques de répartition :")
    for i, (cat_name, count) in enumerate(zip(cat_names, quotas)):
        actual = stats['distribution'].get(cat_name, 0)
        print(f"   • {cat_name:30s} : {actual:6,} / {count:6,}")
    
    print("\n" + "=" * 70)
    
    return assignments, predicted_labels, stats

    
# Exécution
assignments, predicted_labels2, stats = assign_products_to_categories(
    similarity_matrix, 
    df_categories_count,
    top_k=20
)


print(f"\n✅ Assignation terminée !")
print(f"   Exemples : {predicted_labels2[:5]}")

RÉPARTITION OPTIMALE - MIN-COST MAX-FLOW
📊 Données validées :
   • 128,253 produits à répartir
   • 21 catégories disponibles
   • Somme des quotas : 128,253

🔨 Construction du graphe...
   ✓ 128,253 arcs Source → Produits
   • Calcul des 20 meilleures catégories par produit...
   ✓ 2,565,060 arcs Produits → Catégories (Top-20)
   ✓ 21 arcs Catégories → Sink

🚀 Résolution du problème d'optimisation...
✅ Solution optimale trouvée !
   • Coût total : 7,989,294,513
   • Similarité moyenne : 0.3771

📈 Statistiques de répartition :
   • apparel & accessories          : 20,000 / 20,000
   • home & garden                  : 20,000 / 20,000
   • furniture                      : 20,000 / 20,000
   • health & beauty                : 20,000 / 20,000
   • toys & games                   :  9,879 /  9,879
   • electronics                    :  8,094 /  8,094
   • sporting goods                 :  4,674 /  4,674
   • arts & entertainment           :  4,318 /  4,318
   • luggage & bags                

In [22]:
# ==============================================================================
# 1. CONFIGURATION & VÉRIFICATIONS
# ==============================================================================

# Adapter selon les noms réels de tes colonnes dans df_categories_count
COL_NAME_COUNT = 'count'         # Colonne contenant le nombre (quota)
COL_NAME_CATEGORY = 'level_1_name' # Colonne contenant le nom de la catégorie

# Vérification des dimensions
n_products, n_cols_sim = similarity_matrix.shape
n_cats_df = len(df_categories_count)

if n_cols_sim != n_cats_df:
    raise ValueError(f"Erreur : La matrice a {n_cols_sim} colonnes mais le DF a {n_cats_df} catégories.")

# Extraction des quotas sous forme de tableau numpy
quotas = df_categories_count[COL_NAME_COUNT].values.astype(int)

# Vérification de la somme (doit être égale au nombre de produits)
diff_quota = n_products - quotas.sum()
if diff_quota != 0:
    print(f"⚠️ ATTENTION : La somme des quotas ({quotas.sum()}) diffère du nombre de produits ({n_products}).")
    print(f"Correction automatique : Ajout/Retrait de {diff_quota} au quota de la dernière catégorie pour équilibrer.")
    quotas[-1] += diff_quota

# ==============================================================================
# 2. CONSTRUCTION DU GRAPHE (Min-Cost Max-Flow)
# ==============================================================================

smcf = min_cost_flow.SimpleMinCostFlow()

# Indices des nœuds
SOURCE = 0
SINK = n_products + n_cats_df + 1
START_PROD_NODES = 1
START_CAT_NODES = n_products + 1

# Paramètres d'optimisation
# Ne pas demander plus de candidats TOP_K que le nombre réel de catégories.
TOP_K = min(10, n_cats_df)  # On garde au maximum 100, ou moins si moins de catégories
COST_MULTIPLIER = 100000     # Pour convertir les distances float en int

print(f"Construction du graphe pour {n_products} produits et {n_cats_df} catégories...")

# --- A. Arcs Source -> Produits ---
# Tous les produits viennent de la source (Capacité 1, Coût 0)
# Pour optimiser la boucle Python, on utilise des listes
sources = [SOURCE] * n_products
targets = list(range(START_PROD_NODES, START_PROD_NODES + n_products))
capacities = [1] * n_products
costs = [0] * n_products
smcf.add_arcs_with_capacity_and_unit_cost(sources, targets, capacities, costs)

# --- B. Arcs Produits -> Catégories (Optimisation Top-K) ---
# On identifie les K meilleures catégories pour chaque produit pour éviter de créer 4 millions d'arcs
print("Calcul des Top-K candidats...")
# argpartition met les K plus grands indices à la fin de chaque ligne
top_k_indices = np.argpartition(similarity_matrix, -TOP_K, axis=1)[:, -TOP_K:]

print("Création des arcs de coûts...")
# On prépare les listes pour le chargement en masse (batch)
u_list, v_list, cap_list, cost_list = [], [], [], []

for i in range(n_products):
    candidates = top_k_indices[i]
    row_scores = similarity_matrix[i] # Accès plus rapide
    
    for cat_idx in candidates:
        score = row_scores[cat_idx]
        
        # Coût = (1 - Similarité) converti en entier
        # On veut minimiser le coût, donc maximiser la similarité
        int_cost = int((1.0 - score) * COST_MULTIPLIER)
        
        u_list.append(START_PROD_NODES + i)
        v_list.append(START_CAT_NODES + int(cat_idx))
        cap_list.append(1)
        cost_list.append(int_cost)

smcf.add_arcs_with_capacity_and_unit_cost(u_list, v_list, cap_list, cost_list)

# --- C. Arcs Catégories -> Puits ---
# Chaque catégorie a une capacité égale à son QUOTA
u_sink, v_sink, cap_sink, cost_sink = [], [], [], []

for j, quota in enumerate(quotas):
    u_sink.append(START_CAT_NODES + j)
    v_sink.append(SINK)
    cap_sink.append(int(quota))
    cost_sink.append(0)

smcf.add_arcs_with_capacity_and_unit_cost(u_sink, v_sink, cap_sink, cost_sink)

# --- D. Définition du Flux ---
smcf.set_node_supply(SOURCE, int(n_products))
smcf.set_node_supply(SINK, -int(n_products))

# ==============================================================================
# 3. RÉSOLUTION
# ==============================================================================
print("Lancement du solveur OR-Tools...")
status = smcf.solve()

if status == smcf.OPTIMAL:
    print(f"✅ Solution Optimale trouvée ! Coût total : {smcf.optimal_cost()}")
    
    # Récupération des résultats
    final_assignments = np.zeros(n_products, dtype=int)
    
    # On parcourt les arcs pour voir où le flux est passé
    # Note : C'est plus rapide de parcourir les arcs internes que tous les arcs
    for i in range(smcf.num_arcs()):
        if smcf.flow(i) > 0:
            u = smcf.tail(i)
            v = smcf.head(i)
            # On cherche uniquement les arcs Produit -> Catégorie
            if u >= START_PROD_NODES and u < START_CAT_NODES:
                p_idx = u - START_PROD_NODES
                c_idx = v - START_CAT_NODES
                final_assignments[p_idx] = c_idx

    # Traduction Index -> Nom de catégorie
    # On assume que l'index 0 de quotas correspond à l'index 0 de df_categories_count
    cat_names_map = df_categories_count[COL_NAME_CATEGORY].values
    predicted_labels = [cat_names_map[i] for i in final_assignments]
    
    # Ajout au DataFrame original (si tu l'as sous la main)
    # df_nlp['final_category'] = predicted_labels
    
    print("Exemple des 5 premières assignations :", predicted_labels[:5])
    
else:
    print("❌ Pas de solution trouvée. Vérifiez que la somme des quotas est exacte et que Top-K est suffisant.")

Construction du graphe pour 128253 produits et 21 catégories...
Calcul des Top-K candidats...
Création des arcs de coûts...
Lancement du solveur OR-Tools...
✅ Solution Optimale trouvée ! Coût total : 7989336839
Exemple des 5 premières assignations : ['home & garden', 'toys & games', 'media', 'apparel & accessories', 'office supplies']


In [23]:
df_result = pd.DataFrame(data= predicted_labels , columns=["predicted_level_1_cat"])
df_result["hashed_external_id"] = df_nlp["hashed_external_id"].values
df_result.to_csv("data/predicted_level_1.csv")

In [24]:
df_catalog["predicted_label"] = predicted_labels

df_catalog.iloc[:100].to_csv("data/manuel_verif.csv")